# Radiant RAG MCP — Video Summarization Test Notebook

Tests `ingest_video` MCP tool and `VideoSummarizationAgent`.

| | |
|---|---|
| **Transport** | Streamable HTTP |
| **Backend** | ChromaDB |
| **LLM** | Ollama Cloud — set `OLLAMA_API_KEY` in Colab Secrets |
| **VLM** | `Qwen/Qwen2-VL-2B-Instruct` (T4 GPU recommended) |

| # | Section | LLM | VLM |
|---|---------|-----|-----|
| 1 | Install | | |
| 2 | Import check | | |
| 3 | Configuration | | |
| 4 | Helpers | | |
| 5 | Server startup | | |
| 6 | Connection verification | | |
| 7 | Create synthetic test videos | | |
| 8 | Audio detection smoke test | | |
| 9 | Ingest audio video | | |
| 10 | Ingest silent video | | VLM |
| 11 | Search across video content | | |
| 12 | Query with video context | LLM | |
| 13 | Direct API — `brief` summary | LLM | |
| 14 | Direct API — `standard` summary | LLM | |
| 15 | Direct API — `detailed` summary | LLM | |
| 16 | Summary detail levels comparison | LLM | |
| 17 | YouTube URL ingestion | LLM | |
| 18 | Cleanup | | |

## 1  Install

In [ ]:
import subprocess, sys

ACTIVE_BRANCH = 'main'
CONFIG_PATH   = '/content/config.yaml'

# Core RAG package
!pip install -q "radiant-rag-mcp[chroma] @ git+https://github.com/dshipley71/radiant-rag-mcp.git"
!pip install -q --prefer-binary nest_asyncio httpx "fastmcp>=3.0"
!wget -q "https://raw.githubusercontent.com/dshipley71/radiant-rag-mcp/main/config.yaml" -O {CONFIG_PATH}

# Video dependencies
!pip install -q yt-dlp "faster-whisper>=1.0.0" "scenedetect[opencv]>=0.6.3"
!pip install -q ffmpeg-python Pillow numpy opencv-python-headless
!apt-get install -qq -y ffmpeg > /dev/null 2>&1

# Pre-cache embedding / reranking models
!python3 -c "from sentence_transformers import SentenceTransformer; \\
    SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2'); \\
    print('sentence-transformers  OK')"
!python3 -c "from sentence_transformers import CrossEncoder; \\
    CrossEncoder('cross-encoder/ms-marco-MiniLM-L12-v2'); \\
    print('cross-encoder          OK')"

print('Install complete')


## 2  Import check

In [ ]:
import sys

checks = [
    ('radiant_rag_mcp', 'radiant-rag-mcp'),
    ('radiant_rag_mcp.ingestion.video_processor', 'radiant-rag-mcp (video_processor)'),
    ('radiant_rag_mcp.agents.video_summarization', 'radiant-rag-mcp (video_summarization)'),
    ('fastmcp', 'fastmcp'),
    ('nest_asyncio', 'nest_asyncio'),
    ('chromadb', 'chromadb'),
    ('sentence_transformers', 'sentence-transformers'),
    ('yt_dlp', 'yt-dlp'),
    ('faster_whisper', 'faster-whisper'),
    ('cv2', 'opencv-python-headless'),
    ('PIL', 'Pillow'),
    ('scenedetect', 'scenedetect[opencv]'),
]

_missing = []
for module, pkg in checks:
    try:
        __import__(module)
        print(f'  ok  {module}')
    except ImportError:
        print(f'  MISSING  {module}  ->  pip install {pkg}')
        _missing.append(pkg)

if _missing:
    print(f'Missing: {", ".join(_missing)}')
else:
    print('All imports ok')


## 3  Configuration

Video-specific env vars use the prefix `RADIANT_VIDEO_*`.
Summary detail is controlled by `RADIANT_VIDEO_SUMMARIZATION_SUMMARY_DETAIL`.

In [ ]:
import os
try:
    from google.colab import userdata
    _key = userdata.get('OLLAMA_API_KEY') or ''
except Exception:
    _key = os.environ.get('RADIANT_OLLAMA_API_KEY', '')

os.environ['RADIANT_OLLAMA_BASE_URL'] = 'https://ollama.com/v1'
os.environ['RADIANT_OLLAMA_API_KEY']  = _key
os.environ['RADIANT_TRANSPORT']       = 'http'
os.environ['RADIANT_HOST']            = '127.0.0.1'
os.environ['RADIANT_PORT']            = '8765'
os.environ['RADIANT_STORAGE_BACKEND'] = 'chroma'
os.environ['RADIANT_CONFIG_PATH']     = CONFIG_PATH
os.environ['RADIANT_PIPELINE_USE_CRITIC']      = 'false'
os.environ['RADIANT_CITATION_ENABLED']         = 'false'
os.environ['RADIANT_LLM_BACKEND_TIMEOUT']      = '120'
os.environ['RADIANT_LLM_BACKEND_MAX_RETRIES']  = '0'
os.environ['RADIANT_RERANKING_BACKEND_DEVICE'] = 'cuda'
os.environ['RADIANT_EMBEDDING_BACKEND_DEVICE'] = 'cuda'
os.environ['RADIANT_CHUNKING_USE_LLM_CHUNKING']= 'false'
os.environ['RADIANT_RETRIEVAL_DENSE_TOP_K']    = '5'
os.environ['RADIANT_RETRIEVAL_BM25_TOP_K']     = '5'

# VLM (required for silent video)
os.environ['RADIANT_VLM_ENABLED']    = 'true'
os.environ['RADIANT_VLM_MODEL_NAME'] = 'Qwen/Qwen2-VL-2B-Instruct'
os.environ['RADIANT_VLM_DEVICE']     = 'auto'

# Video processor settings
os.environ['RADIANT_VIDEO_WHISPER_MODEL']                = 'base'
os.environ['RADIANT_VIDEO_WINDOW_DURATION_SECONDS']      = '10.0'
os.environ['RADIANT_VIDEO_WINDOW_OVERLAP_SECONDS']       = '2.0'
os.environ['RADIANT_VIDEO_FRAMES_PER_WINDOW']            = '3'
os.environ['RADIANT_VIDEO_ENABLE_SCENE_CHANGE_DETECTION']= 'true'
os.environ['RADIANT_VIDEO_SCENE_CHANGE_THRESHOLD']       = '0.25'
os.environ['RADIANT_VIDEO_FILMSTRIP_TILE_WIDTH']         = '480'
os.environ['RADIANT_VIDEO_FILMSTRIP_TILE_HEIGHT']        = '270'
os.environ['RADIANT_VIDEO_ENABLE_SILENT_VIDEO_ANALYSIS'] = 'true'

# Video summarization
os.environ['RADIANT_VIDEO_SUMMARIZATION_SUMMARY_DETAIL']           = 'standard'
os.environ['RADIANT_VIDEO_SUMMARIZATION_WINDOW_CAPTION_SENTENCES'] = '4'

SERVER_URL  = f"http://{os.environ['RADIANT_HOST']}:{os.environ['RADIANT_PORT']}/mcp"
HAS_LLM_KEY = bool(_key)

print(f'Server URL      : {SERVER_URL}')
print(f'LLM key set     : {HAS_LLM_KEY}')
print(f'VLM model       : {os.environ["RADIANT_VLM_MODEL_NAME"]}')
print(f'Window duration : {os.environ["RADIANT_VIDEO_WINDOW_DURATION_SECONDS"]}s')
print(f'Frames/window   : {os.environ["RADIANT_VIDEO_FRAMES_PER_WINDOW"]}')
print(f'Summary detail  : {os.environ["RADIANT_VIDEO_SUMMARIZATION_SUMMARY_DETAIL"]}')

if not HAS_LLM_KEY:
    print('[NOTE] No LLM key -- LLM cells will be skipped.')
    print('       Add OLLAMA_API_KEY to Colab Secrets (key icon, left sidebar).')


## 4  Helpers

In [ ]:
import asyncio
import json
import logging
import threading
import time
import textwrap
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

import httpx as _httpx
from fastmcp import Client
from fastmcp.exceptions import ToolError


async def _call(tool: str, args: dict = None):
    try:
        async with Client(SERVER_URL) as client:
            raw = await client.call_tool(tool, args or {})
    except ToolError as e:
        return {'status': 'error', 'tool': tool, 'message': str(e)}
    except Exception as e:
        return {'status': 'error', 'tool': tool, 'message': f'{type(e).__name__}: {e}'}
    content = raw.content if hasattr(raw, 'content') else (raw or [])
    if not content:
        return None
    item = content[0]
    text = item.text if hasattr(item, 'text') else str(item)
    try:
        return json.loads(text)
    except (json.JSONDecodeError, ValueError):
        return text


def run(tool: str, args: dict = None):
    result = asyncio.run(_call(tool, args))
    print(json.dumps(result, indent=2, default=str))
    return result


def skip_llm() -> bool:
    if not HAS_LLM_KEY:
        print('[SKIPPED] Set OLLAMA_API_KEY in Colab Secrets.')
        return True
    return False


def assert_ok(result, field: str = None):
    assert result is not None, 'Tool returned no result'
    if isinstance(result, dict):
        if result.get('status') == 'error':
            raise AssertionError(f'Tool error: {result.get("message", result)}')
        if field:
            assert field in result, f'Expected field "{field}" missing: {result}'
    print('[OK]')


async def _wait_for_server(url: str, timeout: int = 120, interval: float = 1.0) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            async with _httpx.AsyncClient() as http:
                await http.get(url, timeout=2.0,
                               headers={'Accept': 'application/json, text/event-stream'})
                return True
        except (_httpx.ConnectError, _httpx.ConnectTimeout, _httpx.ReadTimeout):
            await asyncio.sleep(interval)
    return False


print('Helpers loaded')


## 5  Server startup

In [ ]:
import subprocess, time as _time

_port = int(os.environ['RADIANT_PORT'])
subprocess.run(['fuser', '-k', f'{_port}/tcp'], capture_output=True)
_time.sleep(1.0)

logging.basicConfig(stream=__import__('sys').stderr, level=logging.WARNING, force=True)

_server_ready  = threading.Event()
_server_error  = None
_transport_used = None


def _run_server():
    global _server_error, _transport_used
    try:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        from radiant_rag_mcp.server import mcp
        print('Package  : radiant_rag_mcp  ok')
        _server_ready.set()
        host = os.environ['RADIANT_HOST']
        port = int(os.environ['RADIANT_PORT'])
        for _t in ['http', 'streamable-http']:
            try:
                _transport_used = _t
                mcp.run(transport=_t, host=host, port=port)
                return
            except Exception as _e:
                if 'Unknown transport' in str(_e) or 'unknown transport' in str(_e).lower():
                    continue
                raise
        raise RuntimeError('No supported HTTP transport found')
    except Exception as exc:
        _server_error = exc
        if not _server_ready.is_set():
            _server_ready.set()


_thread = threading.Thread(target=_run_server, name='radiant-mcp', daemon=True)
_thread.start()

if not _server_ready.wait(timeout=30):
    raise TimeoutError('Server thread did not signal ready within 30 s.')
if _server_error:
    raise _server_error

print(f'Waiting for server at {SERVER_URL} ...')
_deadline = time.time() + 90
while time.time() < _deadline:
    if _server_error:
        raise RuntimeError(f'Server thread failed: {_server_error}')
    if asyncio.run(_wait_for_server(SERVER_URL, timeout=5)):
        print(f'Server ready  ->  {SERVER_URL}  (transport: {_transport_used})')
        break
    time.sleep(1)
else:
    raise TimeoutError('Server did not bind within 90 s.')


## 6  Connection verification

In [ ]:
async def _list_tools():
    async with Client(SERVER_URL) as client:
        return await client.list_tools()

tools = asyncio.run(_list_tools())
print(f'{len(tools)} tool(s) registered:')
for t in sorted(tools, key=lambda x: x.name):
    desc = (t.description or '').splitlines()[0]
    print(f'  {t.name:<26}  {desc}')

registered = {t.name for t in tools}
if {'ingest_video'}.issubset(registered):
    print('\ningest_video tool registered  ok')
else:
    print('\n[WARNING] ingest_video NOT registered')


## 7  Create synthetic test videos

Two 30-second videos generated entirely in-process:
- **`audio_video.mp4`** — coloured scene frames + 440 Hz sine audio
- **`silent_video.mp4`** — coloured scene frames, **no audio stream**

In [ ]:
import cv2
import numpy as np
import subprocess
from pathlib import Path

VIDEO_DIR    = Path('/tmp/radiant_video_test')
VIDEO_DIR.mkdir(exist_ok=True)
AUDIO_VIDEO  = str(VIDEO_DIR / 'audio_video.mp4')
SILENT_VIDEO = str(VIDEO_DIR / 'silent_video.mp4')
FPS, DUR, W, H = 10, 30, 640, 360


def _write_raw(path, scene_labels):
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out    = cv2.VideoWriter(path, fourcc, FPS, (W, H))
    colours = [(180,60,60),(60,120,180),(60,160,80),(140,80,160),(200,160,60)]
    fpsc    = (DUR * FPS) // len(scene_labels)
    for si, label in enumerate(scene_labels):
        col = colours[si % len(colours)]
        for fi in range(fpsc):
            t     = (si * fpsc + fi) / FPS
            frame = np.full((H, W, 3), col, dtype=np.uint8)
            cv2.putText(frame, label, (30,60), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 2)
            cv2.putText(frame, f't={t:.1f}s', (30,110), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (220,220,220), 1)
            bx = int((fi/fpsc)*(W-40))+20
            cv2.rectangle(frame,(bx-4,H-25),(bx+4,H-8),(255,255,100),-1)
            out.write(frame)
    out.release()


# Silent
_raw_s = str(VIDEO_DIR / '_raw_silent.mp4')
_write_raw(_raw_s, ['Lab Setup','Calibration','Data Collection','Analysis','Results'])
subprocess.run(['ffmpeg','-y','-i',_raw_s,'-an','-vcodec','libx264','-crf','28',
                SILENT_VIDEO], capture_output=True, check=True)

# Audio
_raw_a = str(VIDEO_DIR / '_raw_audio.mp4')
_write_raw(_raw_a, ['Introduction','Methodology','Experiment','Discussion','Conclusion'])
_wav   = str(VIDEO_DIR / '_audio.wav')
subprocess.run(['ffmpeg','-y','-f','lavfi','-i',f'sine=frequency=440:duration={DUR}',_wav],
               capture_output=True, check=True)
subprocess.run(['ffmpeg','-y','-i',_raw_a,'-i',_wav,'-c:v','libx264','-crf','28',
                '-c:a','aac','-shortest',AUDIO_VIDEO], capture_output=True, check=True)

for f in [_raw_s, _raw_a, _wav]:
    Path(f).unlink(missing_ok=True)

print('Test videos created:')
for vp in [AUDIO_VIDEO, SILENT_VIDEO]:
    print(f'  {Path(vp).name:<25}  {Path(vp).stat().st_size//1024} KB')


## 8  Audio detection smoke test

`VideoProcessor._has_audio()` should return `True` for the audio video and `False` for the silent one.

In [ ]:
from radiant_rag_mcp.ingestion.video_processor import VideoProcessor
from radiant_rag_mcp.config import VideoProcessorConfig

cfg = VideoProcessorConfig()
vp  = VideoProcessor(config=cfg)

for path, expected in [(AUDIO_VIDEO, True), (SILENT_VIDEO, False)]:
    result = vp._has_audio(path)
    ok = 'ok' if result == expected else 'MISMATCH'
    print(f'  {ok}  {Path(path).name:<25}  has_audio={result}  (expected {expected})')


## 9  Ingest audio video  *(Whisper transcription)*

In [ ]:
%%time
print('=== ingest_video (audio) ===')
r = run('ingest_video', {
    'sources': [AUDIO_VIDEO],
    'hierarchical': True,
    'enable_frame_captioning': False,
    'force_frame_analysis': False,
    'summarize': False,
})
assert_ok(r, 'sources_processed')
print(f'Audio sources  : {r.get("audio_sources")}')
print(f'Silent sources : {r.get("silent_sources")}')
print(f'Chunks created : {r.get("chunks_created")}')
print(f'Errors         : {r.get("errors")}')


In [ ]:
print('=== get_index_stats (after audio ingest) ===')
r = run('get_index_stats')
assert_ok(r)


## 10  Ingest silent video  *(VLM frame-window analysis)*

The silent video has no audio stream. `VideoProcessor` detects this and routes to
`_process_silent_video()`, sampling frames, tiling filmstrips, and captioning each
window with the VLM.

> **Runtime:** ~2-5 min on T4 GPU (30 s video, 10 s windows = 3 windows).

In [ ]:
%%time
print('=== ingest_video (silent -- VLM frame windows) ===')
r = run('ingest_video', {
    'sources': [SILENT_VIDEO],
    'hierarchical': True,
    'enable_frame_captioning': False,
    'force_frame_analysis': False,
    'summarize': False,
})
assert_ok(r, 'sources_processed')
print(f'Audio sources  : {r.get("audio_sources")}')
print(f'Silent sources : {r.get("silent_sources")}')
print(f'Chunks created : {r.get("chunks_created")}')
print(f'Errors         : {r.get("errors")}')


In [ ]:
%%time
# Confirm force_frame_analysis flag routes an audio video through the frame path
print('=== ingest_video (audio, force_frame_analysis=True) ===')
r = run('ingest_video', {
    'sources': [AUDIO_VIDEO],
    'hierarchical': False,
    'force_frame_analysis': True,
    'summarize': False,
})
assert_ok(r)
print(f'Silent sources (forced): {r.get("silent_sources")} -- expected 1')


## 11  Search across video content  *(no LLM)*

All video chunks (both `transcript` and `frame_window_captions`) are searchable.
The `content_type` metadata field identifies which path produced each chunk.

In [ ]:
%%time
print('=== search_knowledge: hybrid ===')
r = run('search_knowledge', {
    'query': 'laboratory calibration data collection',
    'mode': 'hybrid',
    'top_k': 6,
})
assert_ok(r)

if isinstance(r, dict) and r.get('results'):
    for hit in r['results'][:4]:
        meta    = hit.get('metadata', {})
        ctype   = meta.get('content_type', '?')
        ts      = meta.get('start_time', meta.get('window_index', '?'))
        source  = Path(meta.get('source', '?')).name
        preview = hit.get('content', '')[:100].replace('\n', ' ')
        print(f'  [{ctype}] t={ts}  {source}')
        print(f'      {preview}...')
        print()


In [ ]:
%%time
print('=== search_knowledge: bm25 ===')
r = run('search_knowledge', {
    'query': 'analysis results screen display',
    'mode': 'bm25',
    'top_k': 5,
})
assert_ok(r)


## 12  Query with video context  `[LLM]`

In [ ]:
%%time
if not skip_llm():
    print('=== query_knowledge (video-grounded) ===')
    r = run('query_knowledge', {
        'question': ('What laboratory procedures are shown in the videos, '
                     'and what stages of an experiment are covered?'),
        'mode': 'hybrid',
    })
    assert_ok(r, 'answer')
    print()
    print('Answer:')
    print('-' * 60)
    print(r['answer'][:800])
    print('-' * 60)
    print(f'Warnings: {r.get("warnings", [])}')


## 13  Direct API — `brief` summary  `[LLM]`

`summary_detail='brief'`:
- Window captions: 3-5 focused sentences
- Chapter summary: 3-5 sentences (1 paragraph)
- Overall summary: 1 paragraph

In [ ]:
%%time
if not skip_llm():
    from radiant_rag_mcp.app import create_app
    from radiant_rag_mcp.agents.video_summarization import VideoSummarizationAgent
    from radiant_rag_mcp.config import VideoSummarizationConfig

    _app = create_app(CONFIG_PATH)
    _llm = _app._orchestrator._llm

    # Retrieve silent video chunks from the vector store
    hits   = _app.search(SILENT_VIDEO, mode='dense', top_k=20, show_results=False)
    chunks = [doc for doc, _ in hits if doc.meta.get('source') == SILENT_VIDEO]
    print(f'Chunks retrieved for silent video: {len(chunks)}')

    if not chunks:
        print('[NOTE] No chunks found -- run section 10 first.')
    else:
        cfg   = VideoSummarizationConfig(summary_detail='brief')
        agent = VideoSummarizationAgent(llm=_llm, config=cfg)
        res   = agent.summarize_video(SILENT_VIDEO, chunks)

        print(f'Title      : {res.title}')
        print(f'is_silent  : {res.is_silent}')
        print(f'Duration   : {res.duration_seconds:.0f}s')
        print(f'Chapters   : {len(res.chapters)}')
        print(f'Key topics : {res.key_topics}')
        print()
        print('Overall summary (brief):')
        print('-' * 60)
        print(res.summary)
        print('-' * 60)
        print(f'Word count : {len(res.summary.split())}')


## 14  Direct API — `standard` summary  `[LLM]`

`summary_detail='standard'`:
- Chapter summary: 1-2 structured paragraphs (5-10 sentences)
- Overall summary: 2-3 paragraphs (Overview + Content + optional Detail)

In [ ]:
%%time
if not skip_llm():
    cfg   = VideoSummarizationConfig(summary_detail='standard')
    agent = VideoSummarizationAgent(llm=_llm, config=cfg)
    res   = agent.summarize_video(SILENT_VIDEO, chunks)

    print('Overall summary (standard):')
    print('-' * 60)
    print(res.summary)
    print('-' * 60)
    print(f'Word count : {len(res.summary.split())}')
    print()
    print('Chapter breakdowns:')
    for ch in res.chapters:
        print(f'  Ch {ch.index+1}: [{ch.start:.0f}s-{ch.end:.0f}s]  {ch.title}')
        lines = ch.summary.split('\n')[:4]
        for ln in lines:
            print(f'    {ln[:100]}')
        print()


## 15  Direct API — `detailed` summary  `[LLM]`

`summary_detail='detailed'`:
- Chapter summary: 2-3 paragraphs (8-15 sentences)
- Overall summary: 3-4 structured paragraphs using the Overview / Content / Detail / Takeaway schema

In [ ]:
%%time
if not skip_llm():
    cfg   = VideoSummarizationConfig(summary_detail='detailed')
    agent = VideoSummarizationAgent(llm=_llm, config=cfg)
    res   = agent.summarize_video(SILENT_VIDEO, chunks)

    print('Overall summary (detailed):')
    print('-' * 60)
    print(res.summary)
    print('-' * 60)
    print(f'Word count : {len(res.summary.split())}')


## 16  Summary detail levels comparison  `[LLM]`

Word-count comparison across all three detail levels.

In [ ]:
%%time
if not skip_llm():
    results = {}
    for detail in ['brief', 'standard', 'detailed']:
        cfg = VideoSummarizationConfig(summary_detail=detail)
        ag  = VideoSummarizationAgent(llm=_llm, config=cfg)
        results[detail] = ag.summarize_video(SILENT_VIDEO, chunks)

    print(f'{"Detail":<10}  {"Overall words":>14}  {"Chapters":>8}  {"Avg ch words":>14}')
    print(f'{"--"*5}  {"--"*7}  {"--"*4}  {"--"*7}')
    for det, res in results.items():
        ow  = len(res.summary.split())
        avg = (sum(len(c.summary.split()) for c in res.chapters)
               // max(1, len(res.chapters)))
        print(f'{det:<10}  {ow:>14,}  {len(res.chapters):>8}  {avg:>14,}')


## 17  YouTube URL ingestion  `[LLM]`

Ingest a short public YouTube video (audio present - Whisper path).
The transcript chunks are immediately queryable.

> **Runtime:** ~60-120 s (download + transcription).

In [ ]:
%%time
if not skip_llm():
    YT_URL = 'https://www.youtube.com/watch?v=aircAruvnKk'
    print(f'=== ingest_video (YouTube) ===')
    r = run('ingest_video', {
        'sources': [YT_URL],
        'hierarchical': True,
        'summarize': True,
    })
    assert_ok(r, 'sources_processed')
    print(f'Chunks created : {r.get("chunks_created")}')
    print(f'Audio sources  : {r.get("audio_sources")}')
    print(f'Errors         : {r.get("errors")}')

    for src, s in r.get('summaries', {}).items():
        print(f'\nSummary for: {src}')
        print(f'  Title    : {s.get("title")}')
        print(f'  is_silent: {s.get("is_silent")}')
        print(f'  Chapters : {len(s.get("chapters", []))}')
        print()
        print(s.get('summary', '')[:600])


## 18  Cleanup

In [ ]:
print('=== clear_index ===')
r = run('clear_index', {'confirm': True})
if isinstance(r, dict):
    if r.get('cleared'):
        print('[OK] Index cleared.')
    elif '_ensure_index' in r.get('message','') or 'clear failed' in r.get('message','').lower():
        print('[OK] Collection dropped (known ChromaDB reinit issue).')
    else:
        raise AssertionError(f'Unexpected clear failure: {r}')

stats = asyncio.run(_call('get_index_stats'))
if isinstance(stats, dict):
    vi = stats.get('health', stats).get('vector_index', {})
    print(f'Vector doc count post-clear: {vi.get("document_count", "?")}')

import shutil
if Path('/tmp/radiant_video_test').exists():
    shutil.rmtree('/tmp/radiant_video_test')
    print('Removed test video dir')
print('Cleanup complete')


## Summary

In [ ]:
print('=' * 62)
print('  Radiant RAG MCP -- Video Summarization')
print('  Run complete')
print('=' * 62)
print()
print('  Video ingestion')
print('    ok  Audio video  -> Whisper transcription -> transcript chunks')
print('    ok  Silent video -> VLM frame windows     -> caption chunks')
print('    ok  force_frame_analysis flag override')
print()
print('  Search and query')
print('    ok  search_knowledge (hybrid/bm25) across video chunks')
print('   [L]  query_knowledge grounded in video content')
print()
print('  VideoSummarizationAgent')
print('   [L]  brief    -- 3-5 sentences / 1 paragraph')
print('   [L]  standard -- 1-2 paragraphs per chapter, 2-3 overall')
print('   [L]  detailed -- 2-3 paragraphs per chapter, 3-4 overall')
print('   [L]  detail level word-count comparison')
print()
print('  YouTube')
print('   [L]  yt-dlp download + transcription + structured summary')
print()
print(f'  [L] = requires OLLAMA_API_KEY')
print(f'  LLM key available : {HAS_LLM_KEY}')
